# Results

Everything below is read from the committed CSVs. No simulation runs in this notebook: if a number is here, it came out of `scripts/benchmark_damping.py` or `scripts/benchmark_mink.py` and can be regenerated by running them again.

Both benchmarks drive the same path from `diffik.trajectory`: the target travels straight out from the home pose to 0.70 m from the shoulder and comes back, over 6 s at a 2 ms timestep. The Panda reaches about 0.85 m when only position is constrained, but this path holds orientation fixed too, and past roughly 0.70 m that pose stops being reachable at all. Sitting on that edge is what makes the damping visible.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent
RESULTS = REPO_ROOT / "results"
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sweep = pd.read_csv(RESULTS / "damping_sweep.csv")
comparison = pd.read_csv(RESULTS / "mink_comparison.csv")

# The first step of every run is an exact zero: sync_target_to_site puts the
# target on the gripper before the trajectory starts. On a log axis that one
# sample stretches the range over forty decades and flattens everything else,
# so the plots start at the second step. Aggregates keep every row.
sweep_plot = sweep[sweep.step > 0]

print(f"sweep       {len(sweep):6d} rows, {sweep.damping.nunique()} damping values")
print(f"comparison  {len(comparison):6d} rows, methods {sorted(comparison.method.unique())}")
sweep.head()

## Summary

One row per damping value, aggregated over the whole run.

In [ ]:
def rms(series):
    return np.sqrt(np.mean(series**2))


summary = (
    sweep.groupby("damping")
    .agg(
        rms_position_error=("position_error", rms),
        rms_orientation_error=("orientation_error", rms),
        peak_dq=("max_dq", "max"),
        clipped_steps=("clipped", "sum"),
        peak_condition_number=("condition_number", "max"),
    )
    .reset_index()
)
summary.to_csv(RESULTS / "damping_summary.csv", index=False)
summary

Three regimes, not one gradient.

- **Under-damped**, up to about `3e-4`. The joint velocity spikes hard at the singularity and clipping kicks in for a handful of steps, but the arm recovers.
- **The failure band**, roughly `5e-4` to `5e-3`. Position RMS jumps by an order of magnitude and the clip count goes from a dozen steps to over a thousand.
- **Well damped**, from about `7e-3`. No clipping and the lowest tracking error, until the damping grows large enough to cost accuracy on its own: `1e-1` tracks worse than `1e-2`.

## Error against time

The aggregate hides *when* things go wrong. The arm is closest to the singularity at the midpoint of the run, t = 3 s.

In [ ]:
# Eleven series against a ten-colour default cycle would give two damping
# values the same colour. A sequential map also puts the legend in the same
# order as the quantity it describes.
DAMPING_VALUES = sorted(sweep.damping.unique())
DAMPING_COLORS = dict(
    zip(DAMPING_VALUES, plt.cm.viridis(np.linspace(0.0, 0.92, len(DAMPING_VALUES))))
)


def plot_by_damping(column, ylabel, logy=True, ax=None):
    ax = ax or plt.gca()
    for damping, group in sweep_plot.groupby("damping"):
        ax.plot(
            group.time,
            group[column],
            linewidth=1.2,
            color=DAMPING_COLORS[damping],
            label=f"{damping:.0e}",
        )
    ax.axvline(3.0, color="black", linestyle=":", linewidth=1)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("time [s]")
    ax.set_ylabel(ylabel)
    return ax


fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
plot_by_damping("position_error", "position error [m]", ax=axes[0])
plot_by_damping("max_dq", "max |dq| [rad/s]", ax=axes[1])
plot_by_damping("condition_number", "cond(J)", ax=axes[2])
axes[0].set_title("Damping sweep over the reach trajectory (dotted line: full extension)")
axes[0].legend(title="damping", fontsize=8, ncol=2)
for ax in axes[:2]:
    ax.set_xlabel("")
fig.tight_layout()
fig.savefig(FIGURES / "damping_sweep.png", dpi=140)
plt.show()

The three panels say different things.

Away from the singularity every damping value below `1e-1` traces the same position error, so the damping is doing nothing visible there. That is the expected behaviour: the `lambda^2 I` term is negligible next to a well-conditioned `J J^T`.

At the crossing the `max |dq|` panel separates them cleanly, and the ordering is monotone: less damping, larger spike.

The condition number panel shows why. `cond(J)` climbs by three to four orders of magnitude as the arm straightens.

## The failure band

The interesting failure is not the velocity spike. It is what happens *after* the arm crosses the singularity.

In [ ]:
BAND_EXAMPLES = [1e-4, 1e-3, 1e-2]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for damping in BAND_EXAMPLES:
    group = sweep_plot[sweep_plot.damping == damping]
    axes[0].plot(group.time, group.position_error, label=f"{damping:.0e}")
axes[0].axvline(3.0, color="black", linestyle=":", linewidth=1)
axes[0].set_yscale("log")
axes[0].set_xlabel("time [s]")
axes[0].set_ylabel("position error [m]")
axes[0].set_title("Before and after the crossing")
axes[0].legend(title="damping", fontsize=8)

axes[1].bar(
    np.arange(len(summary)), summary.clipped_steps, color="tab:red", alpha=0.8
)
axes[1].set_xticks(np.arange(len(summary)))
axes[1].set_xticklabels([f"{d:.0e}" for d in summary.damping], rotation=45)
axes[1].set_xlabel("damping")
axes[1].set_ylabel("clipped steps")
axes[1].set_title("Steps where the command hit a joint limit")

fig.tight_layout()
fig.savefig(FIGURES / "failure_band.png", dpi=140)
plt.show()

In [ ]:
# Where each run is worst, and whether it recovers.
worst = (
    sweep.loc[sweep.groupby("damping").position_error.idxmax()]
    .set_index("damping")[["time", "position_error", "condition_number"]]
    .rename(columns={"time": "worst_at_time", "position_error": "worst_error"})
)
worst["final_error"] = sweep.groupby("damping").position_error.last()
worst

Inside the band the worst moment arrives late in the run, well after the crossing, and the run never recovers: the final error is as large as the worst one. The condition number at that moment is back down in the tens, so the arm is not sitting in a singularity. It is sitting against a joint limit, and clipping is holding it there.

Clipping does not just shorten the commanded step, it points it somewhere else. Once a joint is pinned, the component of the command along that joint disappears and the end-effector is driven in a direction nobody asked for.

## Damped least squares against mink

Same trajectory, same duration, same timestep, same 1 rad/s velocity bound. The DLS runs scale `dq` down to reach that bound after solving; mink receives it, and the joint limits, as constraints of a quadratic program.

In [ ]:
comparison["label"] = np.where(
    comparison.method == "mink",
    "mink",
    "dls " + comparison.damping.map(lambda d: f"{d:.0e}" if pd.notna(d) else ""),
)

comparison_plot = comparison[comparison.step > 0]

head_to_head = (
    comparison.groupby("label")
    .agg(
        rms_position_error=("position_error", rms),
        rms_orientation_error=("orientation_error", rms),
        peak_dq=("max_dq", "max"),
        clipped_steps=("clipped", "sum"),
    )
    .reset_index()
)
head_to_head.to_csv(RESULTS / "mink_summary.csv", index=False)
head_to_head

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for label, group in comparison_plot.groupby("label"):
    style = dict(linewidth=2.0, color="black") if label == "mink" else dict(linewidth=1.2)
    axes[0].plot(group.time, group.position_error, label=label, **style)
    axes[1].plot(group.time, group.max_dq, label=label, **style)

for ax, ylabel, title in zip(
    axes,
    ["position error [m]", "max |dq| [rad/s]"],
    ["Tracking error", "Commanded joint velocity"],
):
    ax.axvline(3.0, color="black", linestyle=":", linewidth=1)
    ax.set_yscale("log")
    ax.set_xlabel("time [s]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES / "dls_vs_mink.png", dpi=140)
plt.show()

In [ ]:
# What each method asks for at the moment of full extension.
crossing = comparison[np.isclose(comparison.time, 3.0)]
crossing.set_index("label")[
    ["position_error", "max_dq", "condition_number", "clipped"]
]

## Conclusion

mink never clips, and it crosses the singularity asking for a joint velocity an order of magnitude smaller than the under-damped DLS run while tracking about as well. Its final error matches the best-tuned DLS run to five decimal places.

The honest summary is not that the QP tracks better. Well-tuned damped least squares tracks marginally better here, and part of even that gap is the posture task mink carries and these DLS runs do not. What the QP buys is that there is no tuning to get wrong. The damped solver reaches the same quality only once you have found a damping above the failure band, and nothing about a run inside that band announces itself until the arm is already jammed against a limit.

Figures written to `results/figures/`, summaries to `results/damping_summary.csv` and `results/mink_summary.csv`, all derived from the two committed per-step CSVs.